In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("victorsabanzagil/polymers")

# print("Path to dataset files:", path)

In [2]:
import os
import tempfile
import subprocess
import pandas as pd
from sklearn.metrics import pairwise_distances

def compute_ph_with_cpp_ripser(X, ripser_path="../../ripser/ripser", max_dim=1, threshold=None):
    """
    Oblicza macierz odległości i deleguje wyliczenie Homologii Persystentnych 
    do binarnej wersji Ripsera w C++.
    """
    print(f"[*] Obliczanie macierzy odległości (Jaccard) dla {X.shape[0]} próbek...")
    # Odległość Jaccarda jest zoptymalizowana pod rzadkie wektory binarne (Morgan fingerprints)
    D = pairwise_distances(X, metric='jaccard')

    print("[*] Zapisywanie dolnej macierzy trójkątnej do pliku tymczasowego...")
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.csv') as tmp:
        for i in range(D.shape[0]):
            # Wymagany format: lower-distance (tylko dolny trójkąt, oddzielony przecinkami)
            row = D[i, :i]
            tmp.write(",".join(map(str, row)) + "\n")
        tmp_path = tmp.name

    print(f"[*] Wywoływanie pliku wykonywalnego: {ripser_path}...")
    cmd = [ripser_path, "--format", "lower-distance", "--dim", str(max_dim)]
    
    if threshold is not None:
        cmd.extend(["--threshold", str(threshold)])
    
    cmd.append(tmp_path)

    try:
        # Odpalenie procesu i zrzut stdout
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        output = result.stdout
    except subprocess.CalledProcessError as e:
        print(f"[!] Błąd wywołania Ripsera: {e.stderr}")
        output = None
    finally:
        # Sprzątanie po pliku z macierzą
        os.remove(tmp_path)

    return output


if __name__ == "__main__":
    file_path = 'polymers_dataset.csv'
    
    print(f"[*] Wczytywanie prawdziwych danych z {file_path}...")
    df = pd.read_csv(file_path)

    # Parsowanie struktury pliku z Kaggle:
    # Pierwsza kolumna to zazwyczaj 'SMILES', ostatnia to 'class'/'label'. 
    # Środkowe 2048 to bity fingerprintu.
    smiles_col = df.columns[0]
    label_col = df.columns[-1]
    feature_cols = df.columns[1:-1]

    # Zabezpieczenie przed eksplozją pamięci dla kompleksu VR.
    # Jeśli algorytm ma poradzić sobie z całością, zakomentuj te dwie linie 
    # i podstaw: df_sampled = df
    SAMPLE_SIZE_PER_CLASS = 400 
    df_sampled = df.groupby(label_col).sample(n=SAMPLE_SIZE_PER_CLASS, random_state=42)

    classes = df_sampled[label_col].unique()

    # Próba klasyfikacji/rozróżnienia topologicznego poszczególnych rodzajów polimerów.
    for cls in classes:
        print(f"\n{'='*50}")
        print(f"Klasa polimeru: {cls}")
        print(f"{'='*50}")

        X_class = df_sampled[df_sampled[label_col] == cls][feature_cols].values

        # Wywołanie natywnego Ripsera w C++
        ph_output = compute_ph_with_cpp_ripser(
            X=X_class,
            ripser_path="../../ripser/ripser",
            max_dim=1,   # Liczymy H0 i H1
            threshold=0.8  # Ustawiamy maksymalny promień (threshold), żeby ograniczyć wagę filtracji
        )

        if ph_output:
            # Wypisujemy tylko nagłówek wyniku, w praktyce tu będziesz 
            # parsować linie by wrzucić je np. w wektoryzację (Persistence Landscapes)
            lines = ph_output.strip().split('\n')
            print("\n".join(lines[:12]))
            print("... (reszta ukryta)")

[*] Wczytywanie prawdziwych danych z polymers_dataset.csv...

Klasa polimeru: 0
[*] Obliczanie macierzy odległości (Jaccard) dla 400 próbek...


/home/bamichal/miniforge3/envs/tda_stable/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


[*] Zapisywanie dolnej macierzy trójkątnej do pliku tymczasowego...
[*] Wywoływanie pliku wykonywalnego: ../../ripser/ripser...
value range: [0,0.984127]
sparse distance matrix with 400 points and 18214/79800 entries
persistence intervals in dim 0:
 [0,0.0246914)
 [0,0.0246914)
 [0,0.0384615)
 [0,0.04)
 [0,0.0408163)
 [0,0.0444444)
 [0,0.045977)
 [0,0.0470588)
 [0,0.0470588)
... (reszta ukryta)


In [5]:
import os
import re
import tempfile
import subprocess
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import rdmolops
from gudhi.representations import Landscape
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def parse_ripser_cpp_output(output_str, max_death=10.0):
    """
    Parsuje surowe wyjście tekstowe C++ Ripsera do słownika list par [birth, death].
    """
    diagrams = {0: [], 1: []}
    current_dim = None
    
    for line in output_str.split('\n'):
        line = line.strip()
        if line.startswith('persistence intervals in dim'):
            current_dim = int(re.search(r'\d+', line).group())
        elif line.startswith('[') and current_dim is not None:
            parts = line.replace('[', '').replace(')', '').split(',')
            birth = float(parts[0])
            death_str = parts[1].strip()
            
            # Czasami klasa zerowa "żyje" w nieskończoność
            death = max_death if death_str == '' else float(death_str)
                
            # Ignorujemy trywialne punkty startowe bez persystencji
            if death > birth:
                if current_dim in diagrams:
                    diagrams[current_dim].append([birth, death])
                    
    return diagrams

def compute_ph_for_molecule(smiles, ripser_path="../../ripser/ripser"):
    """
    Tworzy macierz grafową dla cząsteczki, odpala binarkę Ripsera i zwraca diagram.
    """
# Zabezpieczenie przed intami, NaN-ami i pustymi stringami
    if not isinstance(smiles, str) or not smiles.strip():
        return None
        
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
        
    # Macierz najkrótszych ścieżek w grafie wiązań (ile wiązań od atomu A do atomu B)
    D = rdmolops.GetDistanceMatrix(mol)
    
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.csv') as tmp:
        for i in range(D.shape[0]):
            row = D[i, :i]
            tmp.write(",".join(map(str, row)) + "\n")
        tmp_path = tmp.name
        
    # Wywołanie natywnego Ripsera
    cmd = [ripser_path, "--format", "lower-distance", "--dim", "1", tmp_path]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        diagrams = parse_ripser_cpp_output(result.stdout)
    except subprocess.CalledProcessError:
        diagrams = None
    finally:
        os.remove(tmp_path)
        
    return diagrams

if __name__ == "__main__":
    file_path = 'polymers_dataset.csv'
    df = pd.read_csv(file_path)
    
    label_col = df.columns[-1]
    # smiles_col = df.columns[0] # Zakładamy, że SMILES to pierwsza kolumna
    smiles_col = 'smiles'
    
    # Dla celów pokazowych ograniczamy próbkę, bo liczenie VR dla każdej cząsteczki zajmie chwilę
    # Zmień/usun, gdy zechcesz puścić obliczenia na całej bazie
    df_sample = df.groupby(label_col).sample(n=100, random_state=42)
    
    print("[*] Wyliczanie topologii dla każdego polimeru z osobna...")
    h0_diagrams, h1_diagrams, labels = [], [], []
    
    for idx, row in df_sample.iterrows():
        smiles = row[smiles_col]
        label = row[label_col]
        
        diagrams = compute_ph_for_molecule(smiles, ripser_path="../../ripser/ripser")
        if diagrams is not None:
            # Gudhi oczekuje pustych macierzy wymiaru (0,2), jeśli brak cech w danym wymiarze
            h0 = np.array(diagrams[0]) if len(diagrams[0]) > 0 else np.empty((0,2))
            h1 = np.array(diagrams[1]) if len(diagrams[1]) > 0 else np.empty((0,2))
            
            h0_diagrams.append(h0)
            h1_diagrams.append(h1)
            labels.append(label)

    print("[*] Transformacja do Persistent Landscapes (Wektoryzacja)...")
    # Definiujemy transformatory Gudhi.
    # Num_landscapes=5 (wyciągamy 5 najważniejszych profilów), resolution=100 (długość każdego)
    # Da nam to 500 cech dla H0 i 500 dla H1
    ls_h0 = Landscape(num_landscapes=5, resolution=100)
    ls_h1 = Landscape(num_landscapes=5, resolution=100)
    
    X_h0 = ls_h0.fit_transform(h0_diagrams)
    X_h1 = ls_h1.fit_transform(h1_diagrams)
    
    # Łączymy cechy H0 i H1 dla każdego polimeru
    X_topological = np.hstack((X_h0, X_h1))
    y = np.array(labels)
    
    print(f"[*] Rozmiar wygenerowanej macierzy cech topologicznych: {X_topological.shape}")
    
    # --- ETAP KLASYFIKACJI ---
    print("\n[*] Trenowanie modelu klasyfikacyjnego (Random Forest)...")
    X_train, X_test, y_train, y_test = train_test_split(X_topological, y, test_size=0.2, random_state=42)
    
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    
    print("\n--- RAPORT Z KLASYFIKACJI NA PODSTAWIE CECH TOPOLOGICZNYCH ---")
    print(classification_report(y_test, y_pred))

[*] Wyliczanie topologii dla każdego polimeru z osobna...
[*] Transformacja do Persistent Landscapes (Wektoryzacja)...
[*] Rozmiar wygenerowanej macierzy cech topologicznych: (100, 1000)

[*] Trenowanie modelu klasyfikacyjnego (Random Forest)...

--- RAPORT Z KLASYFIKACJI NA PODSTAWIE CECH TOPOLOGICZNYCH ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [4]:
print(df.columns[:5])

Index(['Unnamed: 0', 'smiles', 'label', '0', '1'], dtype='str')


In [11]:
import os
import re
import tempfile
import subprocess
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import rdmolops
from gudhi.representations import Landscape
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# --- FUNKCJE POMOCNICZE (TDA & I/O) ---

def parse_ripser_cpp_output(output_str, max_death=10.0):
    diagrams = {0: [], 1: []}
    current_dim = None
    
    for line in output_str.split('\n'):
        line = line.strip()
        if line.startswith('persistence intervals in dim'):
            current_dim = int(re.search(r'\d+', line).group())
        elif line.startswith('[') and current_dim is not None:
            parts = line.replace('[', '').replace(')', '').split(',')
            birth = float(parts[0])
            death_str = parts[1].strip()
            
            death = max_death if death_str == '' else float(death_str)
                
            if death > birth:
                if current_dim in diagrams:
                    diagrams[current_dim].append([birth, death])
                    
    return diagrams

def process_single_molecule(smiles, ripser_path="../../ripser/ripser"):
    """
    Funkcja atomowa dla zrównoleglenia: przetwarza jednego SMILESa do diagramu.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return None
        
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
        
    D = rdmolops.GetDistanceMatrix(mol)
    
    # Tworzymy unikalny plik tymczasowy dla każdego procesu
    fd, tmp_path = tempfile.mkstemp(suffix='.csv', text=True)
    try:
        with os.fdopen(fd, 'w') as tmp:
            for i in range(D.shape[0]):
                row = D[i, :i]
                tmp.write(",".join(map(str, row)) + "\n")
                
        cmd = [ripser_path, "--format", "lower-distance", "--dim", "1", tmp_path]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        diagrams = parse_ripser_cpp_output(result.stdout)
    except subprocess.CalledProcessError:
        diagrams = None
    finally:
        os.remove(tmp_path) # Krytyczne: sprzątanie po rdzeniach
        
    return diagrams

# --- GŁÓWNY PIPELINE ---

if __name__ == "__main__":
    file_path = 'polymers_dataset.csv'
    ripser_bin = "../../ripser/ripser"
    
    print(f"[*] Wczytywanie pełnego zbioru danych...")

    # ... [Początek bez zmian] ...
    
    print(f"[*] Wczytywanie pełnego zbioru danych...")
    df = pd.read_csv(file_path)
    
    # 1. TWARDE PRZYPISANIE KOLUMNY
    # Upewnij się, że wpisujesz tu DOKŁADNĄ nazwę kolumny ze stringami chemicznymi
    smiles_col = 'smiles'  
    label_col = 'label'
    
    # 2. ZAMIANA ŚCIEŻKI NA ABSOLUTNĄ DLA WORKERÓW
    ripser_bin = os.path.abspath("../../ripser/ripser")
    
    smiles_list = df[smiles_col].tolist()
    y_raw = df[label_col].tolist()
    
    print(f"[*] Rozpoczynamy wyliczanie kompleksów Ripsa dla {len(smiles_list)} cząsteczek...")
    
    diagrams_list = Parallel(n_jobs=-1)(
        delayed(process_single_molecule)(smiles, ripser_bin) 
        for smiles in tqdm(smiles_list, desc="Ripser C++")
    )
    
    print("\n[*] Oczyszczanie wyników i rzutowanie macierzy...")
    h0_diagrams, h1_diagrams, y_valid = [], [], []
    
    for diag, label in zip(diagrams_list, y_raw):
        if diag is not None:
            h0 = np.array(diag[0]) if len(diag[0]) > 0 else np.empty((0,2))
            h1 = np.array(diag[1]) if len(diag[1]) > 0 else np.empty((0,2))
            h0_diagrams.append(h0)
            h1_diagrams.append(h1)
            y_valid.append(label)

    # 3. DIAGNOSTYKA PRZED WEKTORYZACJĄ
    print(f"[*] Odrzucono próbki z błędami: {len(smiles_list) - len(y_valid)}")
    print(f"[*] Poprawne diagramy przekazywane do Gudhi: {len(h0_diagrams)}")
    
    if len(h0_diagrams) == 0:
        raise RuntimeError("Zatrzymano: Wszystkie diagramy to None! Sprawdź zawartość df[smiles_col] (czy to na pewno tekst?) lub działanie binarki Ripsera.")

    print(f"[*] Transformacja do Persistent Landscapes...")
    # ... [Dalsza część wektoryzacji Gudhi bez zmian] ...


    # df = pd.read_csv(file_path)
    
    # # Automatyczne wykrywanie kolumn (zakładamy schemat z Kaggle)
    # smiles_col = df.columns[0] if 'SMILES' not in df.columns else 'SMILES'
    # label_col = df.columns[-1]
    
    # # Wyciągnięcie surowych danych
    # smiles_list = df[smiles_col].tolist()
    # y_raw = df[label_col].tolist()
    
    # print(f"[*] Rozpoczynamy wyliczanie kompleksów Ripsa dla {len(smiles_list)} cząsteczek...")
    # print(f"[*] Rozdzielanie pracy na wszystkie rdzenie (może to chwilę potrwać)...")
    
    # # Zrównoleglenie (n_jobs=-1 angażuje wszystkie dostępne wątki/rdzenie)
    # diagrams_list = Parallel(n_jobs=-1)(
    #     delayed(process_single_molecule)(smiles, ripser_bin) 
    #     for smiles in tqdm(smiles_list, desc="Ripser C++")
    # )
    
    # # --- FILTROWANIE BŁĘDÓW RDKITA ---
    # print("\n[*] Oczyszczanie wyników i rzutowanie macierzy...")
    # h0_diagrams, h1_diagrams, y_valid = [], [], []
    
    # for diag, label in zip(diagrams_list, y_raw):
    #     if diag is not None:
    #         h0 = np.array(diag[0]) if len(diag[0]) > 0 else np.empty((0,2))
    #         h1 = np.array(diag[1]) if len(diag[1]) > 0 else np.empty((0,2))
    #         h0_diagrams.append(h0)
    #         h1_diagrams.append(h1)
    #         y_valid.append(label)

    # # --- WEKTORYZACJA (PERSISTENT LANDSCAPES) ---
    print(f"[*] Transformacja do Persistent Landscapes dla {len(y_valid)} poprawnych próbek...")
    
    # Możesz poeksperymentować z parametrami:
    # num_landscapes określa ilość warstw głębokości (k)
    # resolution określa na ile punktów dzielimy dziedzinę
    ls_h0 = Landscape(num_landscapes=5, resolution=100)
    ls_h1 = Landscape(num_landscapes=5, resolution=100)
    
    X_h0 = ls_h0.fit_transform(h0_diagrams)
    X_h1 = ls_h1.fit_transform(h1_diagrams)
    
    X_topological = np.hstack((X_h0, X_h1))
    y = np.array(y_valid)
    
    print(f"[*] Wymiar ostatecznej macierzy wejściowej (X): {X_topological.shape}")
    
    # --- MODELOWANIE ---
    print("\n[*] Trenowanie modelu klasyfikacyjnego na pełnych danych...")
    X_train, X_test, y_train, y_test = train_test_split(X_topological, y, test_size=0.2, random_state=42)
    
    # Ponieważ mamy dużo cech i danych, zwiększamy ilość estymatorów
    clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    
    print("\n" + "="*50)
    print(" RZECZYWISTY RAPORT KLASYFIKACJI TDA ")
    print("="*50)
    print(classification_report(y_test, y_pred))

[*] Wczytywanie pełnego zbioru danych...
[*] Wczytywanie pełnego zbioru danych...
[*] Rozpoczynamy wyliczanie kompleksów Ripsa dla 20609 cząsteczek...


Ripser C++: 100%|██████████| 20609/20609 [00:10<00:00, 1918.58it/s]



[*] Oczyszczanie wyników i rzutowanie macierzy...
[*] Odrzucono próbki z błędami: 0
[*] Poprawne diagramy przekazywane do Gudhi: 20609
[*] Transformacja do Persistent Landscapes...
[*] Transformacja do Persistent Landscapes dla 20609 poprawnych próbek...
[*] Wymiar ostatecznej macierzy wejściowej (X): (20609, 1000)

[*] Trenowanie modelu klasyfikacyjnego na pełnych danych...

 RZECZYWISTY RAPORT KLASYFIKACJI TDA 
                 precision    recall  f1-score   support

oligosaccharide       0.91      0.96      0.93      1365
        peptide       0.49      0.35      0.41      1356
        plastic       0.55      0.67      0.61      1401

       accuracy                           0.66      4122
      macro avg       0.65      0.66      0.65      4122
   weighted avg       0.65      0.66      0.65      4122



In [9]:
print("Ostatnie 5 kolumn:", df.columns[-5:].tolist())
print("\nRozkład klas w wybranej kolumnie (", label_col, "):")
print(df[label_col].value_counts())

Ostatnie 5 kolumn: ['2043', '2044', '2045', '2046', '2047']

Rozkład klas w wybranej kolumnie ( 2047 ):
2047
0    20609
Name: count, dtype: int64


In [10]:
print("Pierwsze 5 kolumn:", df.columns[:5].tolist())

Pierwsze 5 kolumn: ['Unnamed: 0', 'smiles', 'label', '0', '1']


In [13]:
# 1. Zaktualizuj i nadpisz wektor Y prawdziwymi klasami
label_col = 'label' 
y = df[label_col].values

# 2. Szybkie ponowne trenowanie (tylko klasyfikator)
print("[*] Trenowanie modelu klasyfikacyjnego na poprawnych etykietach...")
X_train, X_test, y_train, y_test = train_test_split(X_topological, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("\n--- RZECZYWISTY RAPORT KLASYFIKACJI TDA ---")
print(classification_report(y_test, y_pred))

[*] Trenowanie modelu klasyfikacyjnego na poprawnych etykietach...

--- RZECZYWISTY RAPORT KLASYFIKACJI TDA ---
                 precision    recall  f1-score   support

oligosaccharide       0.91      0.96      0.93      1365
        peptide       0.49      0.35      0.41      1356
        plastic       0.55      0.67      0.61      1401

       accuracy                           0.66      4122
      macro avg       0.65      0.66      0.65      4122
   weighted avg       0.65      0.66      0.65      4122

